In [ ]:
"""
Exercice : Classification de chiffres manuscrits (MNIST) avec un réseau de neurones dense.
Objectifs :
- Charger et prétraiter MNIST
- Construire un réseau à deux couches cachées
- Entraîner et évaluer le modèle
- Visualiser les performances et la matrice de confusion
- Effectuer un réglage basique d'hyperparamètres
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# 1. CHARGEMENT ET PRÉTRAITEMENT DU DATASET MNIST

print("1. Chargement et prétraitement du dataset MNIST")


# Chargement
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print(f"Train images shape: {x_train.shape}")
print(f"Train labels shape: {y_train.shape}")
print(f"Test images shape: {x_test.shape}")
print(f"Test labels shape: {y_test.shape}")

# Normalisation [0,1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# One-hot encoding des labels
y_train_oh = to_categorical(y_train, 10)
y_test_oh = to_categorical(y_test, 10)

# Affichage de quelques images avec leurs labels
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title(f"Label: {y_train[i]}")
    plt.axis('off')
plt.suptitle("Exemples d'images du dataset MNIST")
plt.tight_layout()
plt.show()

# 2. CONSTRUCTION DU RÉSEAU DE NEURONES (FULLY CONNECTED)

print("2. Construction du réseau de neurones")


model = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),  # 28x28 -> 784
    layers.Dense(128, activation='relu'),   # première couche cachée
    layers.Dense(64, activation='relu'),    # deuxième couche cachée
    layers.Dense(10, activation='softmax')  # couche de sortie (10 classes)
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# 3. ENTRAÎNEMENT DU RÉSEAU

print("3. Entraînement du modèle (10 époques)")


history = model.fit(x_train, y_train_oh,
                    batch_size=32,
                    epochs=10,
                    validation_split=0.1,
                    verbose=1)

# 4. ÉVALUATION DES PERFORMANCES

print("4. Évaluation du modèle sur l'ensemble de test")


test_loss, test_acc = model.evaluate(x_test, y_test_oh, verbose=0)
print(f"Précision sur le test : {test_acc:.4f}")
print(f"Perte sur le test : {test_loss:.4f}")

# Courbes d'apprentissage
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train accuracy')
plt.plot(history.history['val_accuracy'], label='Validation accuracy')
plt.title('Accuracy over epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.title('Loss over epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

# Prédictions sur le test
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(10), yticklabels=range(10))
plt.title('Matrice de confusion - Chiffres MNIST')
plt.xlabel('Prédiction')
plt.ylabel('Vérité terrain')
plt.show()

# Identification des chiffres les plus difficiles
# On calcule la précision par classe
class_acc = cm.diagonal() / cm.sum(axis=1)
print("\nPrécision par classe :")
for i in range(10):
    print(f"Classe {i}: {class_acc[i]:.4f}")

worst_class = np.argmin(class_acc)
print(f"\nLa classe la plus difficile à reconnaître est {worst_class} avec une précision de {class_acc[worst_class]:.4f}")

# Affichage de quelques erreurs de classification
errors = np.where(y_test != y_pred_classes)[0]
print(f"\nNombre total d'erreurs : {len(errors)} sur {len(y_test)} images.")
n_show = min(12, len(errors))
plt.figure(figsize=(12, 4))
for i, idx in enumerate(errors[:n_show]):
    plt.subplot(2, 6, i+1)
    plt.imshow(x_test[idx], cmap='gray')
    plt.title(f"True: {y_test[idx]}\nPred: {y_pred_classes[idx]}")
    plt.axis('off')
plt.suptitle("Exemples d'images mal classifiées")
plt.tight_layout()
plt.show()

# 5. EXPÉRIENCE DE RÉGLAGE D'HYPERPARAMÈTRES (basique)

print("5. Réglage d'hyperparamètres : comparaison de différentes architectures")


# On définit quelques configurations à tester
configs = [
    {"name": "Petit réseau", "layers": [64, 32], "learning_rate": 0.001},
    {"name": "Grand réseau", "layers": [256, 128, 64], "learning_rate": 0.001},
    {"name": "Taux d'apprentissage élevé", "layers": [128, 64], "learning_rate": 0.01}
]

results = []
for cfg in configs:
    print(f"\nTest de la configuration : {cfg['name']}")
    model_tmp = models.Sequential([layers.Flatten(input_shape=(28,28))])
    for units in cfg["layers"]:
        model_tmp.add(layers.Dense(units, activation='relu'))
    model_tmp.add(layers.Dense(10, activation='softmax'))

    opt = optimizers.Adam(learning_rate=cfg["learning_rate"])
    model_tmp.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

    # Entraînement rapide sur 5 époques pour comparaison
    history_tmp = model_tmp.fit(x_train, y_train_oh, epochs=5, batch_size=32,
                                validation_split=0.1, verbose=0)
    val_acc = history_tmp.history['val_accuracy'][-1]
    results.append((cfg["name"], val_acc))
    print(f"  Précision validation (dernière époque) : {val_acc:.4f}")

print("\nRésumé des performances des différentes configurations :")
for name, acc in results:
    print(f"{name}: {acc:.4f}")


print("Exercice terminé avec succès.")
